# Model Comparison: Information Extraction Capabilities

Compares field extraction accuracy (F1) and throughput across vision-language models.

**Configuration:** Edit `comparison_config.yaml` to change models, fields, or display settings.

**Requirements:** `pip install pyyaml pandas numpy matplotlib seaborn`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from comparison_utils import (
    load_config,
    load_ground_truth,
    load_results,
    evaluate_model,
    plot_overall_f1,
    plot_field_heatmap,
    plot_throughput,
)

sns.set_theme(style="whitegrid", font_scale=1.1)

CONFIG_PATH = Path("comparison_config.yaml")
config = load_config(CONFIG_PATH)
base_path = CONFIG_PATH.parent

print(
    f"Loaded config: {len(config['models'])} models, "
    f"{len(config['fields']['scalar'])} scalar fields, "
    f"{len(config['fields']['list'])} list fields"
)

In [ ]:
gt = load_ground_truth(base_path / config["ground_truth"]["path"])
print(f"Ground truth: {len(gt)} images")

model_results = {}
for key, model_cfg in config["models"].items():
    model_results[key] = {
        "config": model_cfg,
        "data": load_results(base_path / model_cfg["results_path"]),
    }
    print(f"  {model_cfg['display_name']}: {len(model_results[key]['data'])} images")

In [ ]:
scores_dfs = []
throughput_rows = []

for key, mr in model_results.items():
    display_name = mr["config"]["display_name"]

    df = evaluate_model(
        mr["data"],
        gt,
        config["fields"]["scalar"],
        config["fields"]["list"],
    )
    df["model"] = display_name
    scores_dfs.append(df)

    times = [
        v["processing_time"] for v in mr["data"].values() if v["processing_time"] > 0
    ]
    median_time = pd.Series(times).median() if times else float("inf")
    throughput_rows.append(
        {
            "model": display_name,
            "docs_per_min": 60.0 / median_time if median_time > 0 else 0.0,
        }
    )

all_scores = pd.concat(scores_dfs, ignore_index=True)

# Per-field F1 pivot: rows=fields, columns=models, values=mean score
field_df = all_scores.pivot_table(
    index="field", columns="model", values="score", aggfunc="mean"
)

# Summary: per-model mean F1 and std
summary_df = all_scores.groupby("model")["score"].agg(["mean", "std"]).reset_index()
summary_df.columns = ["model", "mean_f1", "std_f1"]

throughput_df = pd.DataFrame(throughput_rows)

print("Evaluation complete.")
print(summary_df.to_string(index=False))

In [ ]:
# Styled per-field F1 table
styled = (
    field_df.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1)
    .format("{:.3f}")
    .set_caption("Per-Field F1 Scores by Model")
)
display(styled)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=tuple(config["display"]["figure_size"]))
colors = {cfg["display_name"]: cfg["color"] for _, cfg in config["models"].items()}

plot_overall_f1(summary_df, colors, axes[0])
plot_field_heatmap(field_df, axes[1])
plot_throughput(throughput_df, colors, axes[2])

fig.suptitle(
    "Model Comparison: Information Extraction Capabilities",
    fontweight="bold",
    fontsize=14,
    y=1.02,
)
fig.tight_layout()

output_dir = base_path / config["display"]["output_dir"]
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(
    output_dir / f"comparison.{config['display']['save_format']}",
    dpi=config["display"]["dpi"],
    bbox_inches="tight",
)
print(f"Dashboard saved to {output_dir}")
plt.show()

In [ ]:
output_dir = base_path / config["display"]["output_dir"]
field_df.to_csv(output_dir / "per_field_f1.csv")
summary_df.to_csv(output_dir / "model_summary.csv", index=False)
throughput_df.to_csv(output_dir / "throughput.csv", index=False)
print(f"CSVs saved to {output_dir}")